# Data Cleaning Workflow — DaimonUpDown Workforce Project

## Purpose

This notebook documents the database loading, data-quality validation, cleaning decisions, troubleshooting, and final verification performed for the DaimonUpDown workforce project.

The cleaning workflow was performed primarily with **MariaDB, SQL, and terminal commands**.

Python is not used as the primary data-cleaning tool in this workflow. Python utilities used later for dashboard file conversion are documented separately.


## 1. Data Cleaning Process

The overall process was:

**Raw CSV files → MariaDB loading → Identify loading problems → Correct SQL loading logic → Reload → Data-quality checks → Validate final database**

The goal was to preserve the source information without inventing values.

## 2. Database and Source Structure

Database:

`daimonupdown`

The project contains employee, employment, assignment, recruitment, performance, attendance, training, onboarding, offboarding, and related workforce tables.

The cleaned database is later used to create Tableau-ready datasets for the Workforce and Employee Retention dashboards.

## 3. Problems Found During Loading

### Onboarding

- Empty `onboarding_end_date` values caused date-truncation warnings.
- Empty `onboarding_satisfaction` values caused incorrect-integer warnings.
- The loading logic was corrected so empty values become SQL `NULL` where appropriate.

### Offboarding

- Boolean values arrived from CSV as `True` and `False`.
- Empty boolean and numeric values caused additional loading warnings.
- The loading logic was corrected to convert these values before insertion.

### General date handling

Empty CSV date values should not be inserted as empty strings into typed date columns.

The `NULLIF(value, '')` pattern was used where appropriate to convert empty CSV values to SQL `NULL`.

## 4. Onboarding Cleaning

The corrected loading process uses variables and conversion logic for nullable fields.

Example SQL pattern:

In [ ]:
SET onboarding_end_date = NULLIF(@onboarding_end_date, '');

Empty satisfaction values were also converted appropriately instead of forcing an empty string into an integer column.

### Validation result

Final onboarding row count: **1,500**.

## 5. Offboarding Cleaning

The source contained boolean values such as:

In [ ]:
True
False

These values were normalized to the database representation and empty values were converted to `NULL` where the schema allowed it.

### Validation result

Final offboarding row count: **320**.

## 6. Database Row-Count Validation

After the corrected loading process, table counts were checked to confirm that the expected records were present.

In [ ]:
sudo mariadb -e "
USE daimonupdown;
SELECT 'applications' AS table_name, COUNT(*) AS row_count FROM applications
UNION ALL SELECT 'assignments', COUNT(*) FROM assignments
UNION ALL SELECT 'attendance', COUNT(*) FROM attendance
UNION ALL SELECT 'candidates', COUNT(*) FROM candidates
UNION ALL SELECT 'client_feedback', COUNT(*) FROM client_feedback
UNION ALL SELECT 'clients', COUNT(*) FROM clients
UNION ALL SELECT 'compensation_history', COUNT(*) FROM compensation_history
UNION ALL SELECT 'departments', COUNT(*) FROM departments
UNION ALL SELECT 'employees', COUNT(*) FROM employees
UNION ALL SELECT 'employee_surveys', COUNT(*) FROM employee_surveys
UNION ALL SELECT 'employment_history', COUNT(*) FROM employment_history
UNION ALL SELECT 'locations', COUNT(*) FROM locations
UNION ALL SELECT 'offboarding', COUNT(*) FROM offboarding
UNION ALL SELECT 'onboarding', COUNT(*) FROM onboarding
UNION ALL SELECT 'performance', COUNT(*) FROM performance
UNION ALL SELECT 'positions', COUNT(*) FROM positions
UNION ALL SELECT 'qualifications', COUNT(*) FROM qualifications
UNION ALL SELECT 'schools', COUNT(*) FROM schools
UNION ALL SELECT 'training', COUNT(*) FROM training
UNION ALL SELECT 'visa_history', COUNT(*) FROM visa_history;
"

### Final validated counts

| Table | Rows |
|---|---:|
| applications | 2,600 |
| assignments | 2,341 |
| attendance | 22,567 |
| candidates | 2,000 |
| client_feedback | 5,895 |
| clients | 300 |
| compensation_history | 10 |
| departments | 9 |
| employees | 1,600 |
| employee_surveys | 11,956 |
| employment_history | 1,600 |
| locations | 486 |
| offboarding | 320 |
| onboarding | 1,500 |
| performance | 3,029 |
| positions | 6 |
| qualifications | 1,904 |
| schools | 486 |
| training | 9,738 |
| visa_history | 1,500 |

## 7. Foreign-Key Relationship Checks

The database relationships were inspected through `INFORMATION_SCHEMA`.

In [ ]:
sudo mariadb -e "
SELECT
    TABLE_NAME,
    COLUMN_NAME,
    REFERENCED_TABLE_NAME,
    REFERENCED_COLUMN_NAME,
    CONSTRAINT_NAME
FROM INFORMATION_SCHEMA.KEY_COLUMN_USAGE
WHERE TABLE_SCHEMA = 'daimonupdown'
  AND REFERENCED_TABLE_NAME IS NOT NULL
ORDER BY TABLE_NAME, COLUMN_NAME;
"

## 8. Orphan-Record Checks

These checks verify that child records reference existing employees.

In [ ]:
sudo mariadb -e "
USE daimonupdown;

SELECT COUNT(*) AS orphaned_assignments
FROM assignments a
LEFT JOIN employees e ON a.employee_id = e.employee_id
WHERE e.employee_id IS NULL;

SELECT COUNT(*) AS orphaned_training
FROM training t
LEFT JOIN employees e ON t.employee_id = e.employee_id
WHERE e.employee_id IS NULL;

SELECT COUNT(*) AS orphaned_performance
FROM performance p
LEFT JOIN employees e ON p.employee_id = e.employee_id
WHERE e.employee_id IS NULL;

SELECT COUNT(*) AS orphaned_attendance
FROM attendance a
LEFT JOIN employees e ON a.employee_id = e.employee_id
WHERE e.employee_id IS NULL;
"

### Validation result

The checked orphan-record queries returned **0**.

## 9. Duplicate Primary-Key Checks

In [ ]:
sudo mariadb -e "
USE daimonupdown;
SELECT
    'employees' AS table_name,
    COUNT(*) - COUNT(DISTINCT employee_id) AS duplicate_ids
FROM employees;
"

### Validation result

The duplicate-ID checks returned **0**.

## 10. Invalid Date Checks

In [ ]:
sudo mariadb -e "
USE daimonupdown;
SELECT COUNT(*) AS invalid_client_dates
FROM clients
WHERE client_relationship_end_date IS NOT NULL
  AND client_relationship_start_date > client_relationship_end_date;

SELECT COUNT(*) AS invalid_assignment_dates
FROM assignments
WHERE assignment_end_date IS NOT NULL
  AND assignment_start_date > assignment_end_date;

SELECT COUNT(*) AS invalid_employment_dates
FROM employment_history
WHERE employment_end_date IS NOT NULL
  AND employment_start_date > employment_end_date;

SELECT COUNT(*) AS invalid_onboarding_dates
FROM onboarding
WHERE onboarding_end_date IS NOT NULL
  AND onboarding_start_date > onboarding_end_date;
"

These checks identify impossible date relationships such as an end date occurring before a start date.

## 11. NULL Checks for Important Fields

In [ ]:
sudo mariadb -e "
USE daimonupdown;
SELECT 'employees.employee_id' AS field_name, COUNT(*) - COUNT(employee_id) AS null_count FROM employees
UNION ALL SELECT 'employees.candidate_id', COUNT(*) - COUNT(candidate_id) FROM employees
UNION ALL SELECT 'assignments.employee_id', COUNT(*) - COUNT(employee_id) FROM assignments
UNION ALL SELECT 'assignments.client_id', COUNT(*) - COUNT(client_id) FROM assignments
UNION ALL SELECT 'performance.employee_id', COUNT(*) - COUNT(employee_id) FROM performance
UNION ALL SELECT 'attendance.employee_id', COUNT(*) - COUNT(employee_id) FROM attendance;
"

The checked key fields contained **0 unexpected NULLs**.

## 12. Category and Status Checks

In [ ]:
sudo mariadb -e "
USE daimonupdown;
SELECT DISTINCT client_status FROM clients ORDER BY client_status;
SELECT DISTINCT assignment_status FROM assignments ORDER BY assignment_status;
SELECT DISTINCT employment_status FROM employment_history ORDER BY employment_status;
SELECT DISTINCT candidate_status FROM candidates ORDER BY candidate_status;
SELECT DISTINCT completion_status FROM training ORDER BY completion_status;
"

These queries verify that categorical fields contain expected values before they are used in analysis and Tableau filters.

## 13. Cleaning Decisions

| Problem | Action | Reason |
|---|---|---|
| Empty date | Convert empty string to SQL `NULL` | Preserve nullable date semantics |
| Empty integer | Convert to SQL `NULL` where nullable | Avoid invalid integer values |
| `True` / `False` strings | Normalize to database boolean/integer values | Match database schema |
| Duplicate IDs | Investigate and validate | Protect primary-key integrity |
| Orphan records | Check foreign-key relationships | Protect relational integrity |

The guiding principle was:

**Clean the data without inventing values.**

Missing source information remains missing (`NULL`) rather than being guessed.

## 14. Troubleshooting

### Empty date warnings

Empty CSV date values caused warnings when loaded into typed date columns. The loading logic was changed to convert empty values to SQL `NULL` where appropriate.

### Empty numeric values

Empty values in numeric fields were handled as `NULL` where the database schema permitted it rather than inserting empty strings.

### Boolean values

CSV boolean values such as `True` and `False` required normalization before insertion into the database representation.

### SQL vs Bash

SQL commands must be passed to MariaDB. They should not be typed directly at the Bash prompt.

Example:

```text
sudo mariadb -e "SELECT COUNT(*) FROM employees;"
```

### MariaDB service

If MariaDB cannot connect through `/run/mysqld/mysqld.sock`, check whether the service is running:

In [ ]:
sudo systemctl status mariadb

# Start MariaDB if it is inactive
sudo systemctl start mariadb

## 15. Reproducibility

The database loading and validation logic is stored in the project's SQL directory:

```text
sql/03_load_data.sql
sql/04_data_quality_checks.sql
sql/06_onboarding_offboarding.sql
```

This notebook explains what was cleaned, why it was cleaned, and how the data was validated. The SQL files contain the reproducible database operations.

## 16. Final Status

The MariaDB database was successfully loaded and validated after correcting the identified onboarding and offboarding data-loading issues.

The validated database provides the source data used by the project's Tableau dashboards, including the Workforce Dashboard and Employee Retention Dashboard.

The next step for dashboard work is to use the validated data to create Tableau-ready datasets and visualizations.